# Pré-processamento para Content-Based Filtering

Neste notebook, prepararemos os dados para o algoritmo de recomendação. Focaremos no tratamento dos gêneros literários e na normalização das variáveis numéricas.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Carregamento dos dados
df = pd.read_parquet('../../datasets/books.parquet')
print(f"Dataset carregado. Shape: {df.shape}")

## Padronização e Limpeza
Nesta etapa, tratamos a coluna de gêneros (convertendo para minúsculas e dividindo em listas) e garantimos que as colunas numéricas estejam corretas.

In [ ]:
# Padronizar o campo genre: lowercase, split por vírgula, e remover espaços extras
df['genre_list'] = df['genre'].fillna('').astype(str).str.lower().str.split(',')
df['genre_list'] = df['genre_list'].apply(lambda x: [g.strip() for g in x if g.strip()])

# Converter rating, pages e totalratings para numérico (forçando erros a virarem NaN)
cols_numericas = ['rating', 'pages', 'totalratings']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Preencher nulos nas colunas numéricas com a mediana de cada uma
for col in cols_numericas:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)

print("Limpeza concluída!")

## Binarização e Normalização
Aplicamos o `MultiLabelBinarizer` para criar uma matriz onde cada gênero vira uma coluna binária. Em seguida, normalizamos as variáveis numéricas para que fiquem na mesma escala (0 a 1) usando o `MinMaxScaler`.

In [ ]:
# Aplicar MultiLabelBinarizer no genre
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
print(f"Matriz de gêneros criada. Shape: {genre_matrix.shape}")

# Normalizar as colunas numéricas com MinMaxScaler (0 a 1)
scaler = MinMaxScaler()
numeric_matrix = scaler.fit_transform(df[cols_numericas])
print(f"Matriz numérica normalizada. Shape: {numeric_matrix.shape}")

## Criação da Matriz Final e Divisão de Treino/Teste
Com as matrizes de gênero e numéricas prontas, nós as concatenamos em uma única matriz `X`. Em seguida, fazemos a divisão salvando os índices (útil para Content-Based Filtering, pois rastrearemos os livros pelos índices).

In [ ]:
# Concatenar gênero binarizado + numéricos normalizados com np.hstack
X = np.hstack((genre_matrix, numeric_matrix))
print(f"Matriz X final consolidada. Shape: {X.shape}")

# Fazer train_test_split salvando os índices (random_state fixo conforme instrução do professor)
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

print(f"Índices de Treino: {len(train_idx)} | Índices de Teste: {len(test_idx)}")

# Opcional: Para evitar processar isso toda vez, podemos exportar as matrizes
# np.save('../../datasets/X_features.npy', X)
# np.save('../../datasets/train_idx.npy', train_idx)
# np.save('../../datasets/test_idx.npy', test_idx)

In [ ]:
# Criação das pastas caso não existam
import os
import joblib

os.makedirs('../../Machine Learning/data/processed', exist_ok=True)
os.makedirs('../../Machine Learning/models', exist_ok=True)

# 1. Salvar dataset limpo (usaremos parquet para ser mais rápido e menor que CSV)
df.to_parquet('../../Machine Learning/data/processed/books_clean.parquet', index=False)

# 2. Salvar matriz X compactada (pois 84 mil linhas x 1100 colunas pode ficar muito pesado para o GitHub se for .npy normal)
np.savez_compressed('../../Machine Learning/data/processed/X_matrix.npz', X=X)

# 3. Salvar os índices de treino e teste
np.save('../../Machine Learning/data/processed/train_idx.npy', train_idx)
np.save('../../Machine Learning/data/processed/test_idx.npy', test_idx)

# 4. Salvar os objetos transformadores (MLB e Scaler)
joblib.dump(mlb, '../../Machine Learning/models/mlb.pkl')
joblib.dump(scaler, '../../Machine Learning/models/scaler.pkl')

print("Artefatos salvos com sucesso nas pastas 'data/processed' e 'models'!")